In [0]:
-- LBE Concession Revenue by Week (starting Jan 2 2023) — pivoted by Brand_Tier
-- Date range: Jan 2, 2023 – Mar 31, 2026 | ConcessionFLAG = 'Yes'
-- Weeks align to ISO Monday; Jan 2 2023 is a Monday so first week = 2023-01-02

WITH lbe_lookup AS (
    SELECT *
    FROM (VALUES
        -- TRR HPL
        (7002, 'TRR_HPL'), (2001, 'TRR_HPL'), (3001, 'TRR_HPL'),
        (3002, 'TRR_HPL'), (1001, 'TRR_HPL'),
        -- TRR LPL
        (7009, 'TRR_LPL'), (3003, 'TRR_LPL'), (7003, 'TRR_LPL'),
        (8001, 'TRR_LPL'), (7014, 'TRR_LPL'),
        -- TRR NL
        (9003, 'TRR_NL'), (1002, 'TRR_NL'),
        -- PDM NL
        (7021, 'PDM_NL'), (7020, 'PDM_NL'),
        -- PDM OL
        (7012, 'PDM_OL'), (7010, 'PDM_OL'), (5002, 'PDM_OL'),
        (7001, 'PDM_OL')
    ) AS t(location_id, brand_tier)
),

filtered_txns AS (
    SELECT
        CAST(DATE_TRUNC('WEEK', TO_DATE(CAST(VisitDateId AS STRING), 'yyyyMMdd')) AS DATE) AS week_start,
        l.brand_tier,
        t.Net_Revenue,
        t.Transaction_Number
    FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
    JOIN lbe_lookup l ON t.LocationId = l.location_id
    WHERE t.VisitDateId >= 20230102
      AND t.VisitDateId <= 20260801
      AND t.ConcessionFLAG = 'Yes'
)

SELECT
    week_start,
    -- Revenue by tier
    ROUND(SUM(CASE WHEN brand_tier = 'TRR_HPL' THEN Net_Revenue END), 2) AS TRR_HPL,
    ROUND(SUM(CASE WHEN brand_tier = 'TRR_LPL' THEN Net_Revenue END), 2) AS TRR_LPL,
    ROUND(SUM(CASE WHEN brand_tier = 'TRR_NL'  THEN Net_Revenue END), 2) AS TRR_NL,
    ROUND(SUM(CASE WHEN brand_tier = 'PDM_NL'  THEN Net_Revenue END), 2) AS PDM_NL,
    ROUND(SUM(CASE WHEN brand_tier = 'PDM_OL'  THEN Net_Revenue END), 2) AS PDM_OL,
    ROUND(SUM(Net_Revenue), 2)                                            AS Total,
    -- Transaction counts by tier
    COUNT(DISTINCT CASE WHEN brand_tier = 'TRR_HPL' THEN Transaction_Number END) AS TRR_HPL_Txns,
    COUNT(DISTINCT CASE WHEN brand_tier = 'TRR_LPL' THEN Transaction_Number END) AS TRR_LPL_Txns,
    COUNT(DISTINCT CASE WHEN brand_tier = 'TRR_NL'  THEN Transaction_Number END) AS TRR_NL_Txns,
    COUNT(DISTINCT CASE WHEN brand_tier = 'PDM_NL'  THEN Transaction_Number END) AS PDM_NL_Txns,
    COUNT(DISTINCT CASE WHEN brand_tier = 'PDM_OL'  THEN Transaction_Number END) AS PDM_OL_Txns,
    COUNT(DISTINCT Transaction_Number)                                            AS Total_Txns
FROM filtered_txns
GROUP BY week_start
ORDER BY week_start